# Diffusion Model (Flow Matching) Tutorial on 2D Two Moons

**Companion to the Flow Map Tutorial — same dataset, same network, same conventions.**

This notebook trains a standard **diffusion model** via **Conditional Flow Matching** (CFM)
and samples with **Euler** and **Heun's** (2nd-order) ODE solvers.

### Setup

**Forward process** (linear interpolation, same as the flow map tutorial):

$$\mathbf{x}_s = (1 - s)\,\mathbf{x}_0 + s\,\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}), \qquad s \in [0, 1]$$

- $s = 0$: clean data $\mathbf{x}_0$
- $s = 1$: pure noise $\boldsymbol{\epsilon}$

**Conditional velocity** (constant along each straight path):

$$\mathbf{v}_{\text{cond}} = \frac{\mathrm{d}\mathbf{x}_s}{\mathrm{d}s} = \boldsymbol{\epsilon} - \mathbf{x}_0$$

**The model** learns a velocity network $\mathbf{v}_\theta(\mathbf{x}_s, s)$ that approximates
the *marginal* velocity field $\mathbf{v}^*(\mathbf{x}_s, s) = \mathbb{E}[\boldsymbol{\epsilon} - \mathbf{x}_0 \mid \mathbf{x}_s]$.

### Training vs. Sampling

| | Training | Sampling |
|:---|:---|:---|
| **What** | Regress $\mathbf{v}_\theta$ against conditional velocity | Solve the ODE backward from $s\!=\!1$ to $s\!=\!0$ |
| **Loss** | $\|\mathbf{v}_\theta(\mathbf{x}_s, s) - (\boldsymbol{\epsilon} - \mathbf{x}_0)\|^2$ | Euler or Heun's method |
| **# steps** | 1 gradient step per sample | $N$ function evaluations (NFE) |

## 0. Setup & Imports

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Dataset & Forward Process

Same two-moons dataset and forward process as the flow map tutorial:

$$\mathbf{x}_s = (1 - s)\,\mathbf{x}_0 + s\,\boldsymbol{\epsilon}$$

In [ ]:
def make_dataset(n_samples=100_000, noise=0.05):
    """Generate two-moons with modes well separated (same as flow map tutorial)."""
    X, _ = make_moons(n_samples=n_samples, noise=noise, random_state=SEED)
    X = X * 2; X[:, 1] -= 0.5
    return torch.tensor(X, dtype=torch.float32)

def forward_process(x_0, eps, s):
    """x_s = (1-s)*x_0 + s*eps"""
    return (1.0 - s) * x_0 + s * eps

dataset = make_dataset(100_000)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(dataset[:5000, 0], dataset[:5000, 1], s=1, alpha=0.3)
ax.set_title('Two Moons (scaled)'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 2. Network Architecture

Same sinusoidal-time-embedding MLP as the flow map tutorial.
The diffusion model has a single time condition $s$.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10_000) * torch.arange(half, device=t.device, dtype=t.dtype) / half)
        args = t.unsqueeze(-1) * freqs
        return torch.cat([args.sin(), args.cos()], dim=-1)

class MLP(nn.Module):
    def __init__(self, data_dim=2, n_time_conds=1, hidden_dim=256, n_layers=4, time_emb_dim=64):
        super().__init__()
        self.time_embs = nn.ModuleList([SinusoidalTimeEmbedding(time_emb_dim) for _ in range(n_time_conds)])
        in_dim = data_dim + time_emb_dim * n_time_conds
        layers = []
        for i in range(n_layers):
            layers.append(nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim))
            layers.append(nn.SiLU())
        layers.append(nn.Linear(hidden_dim, data_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x, *times):
        embs = [fn(t) for fn, t in zip(self.time_embs, times)]
        return self.net(torch.cat([x] + embs, dim=-1))

## 3. Diffusion Model — Conditional Flow Matching

### Training objective (CFM loss)

Given $\mathbf{x}_0 \sim p_{\text{data}}$, $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, $s \sim p(s)$:

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{\mathbf{x}_0, \boldsymbol{\epsilon}, s}\!\left[\left\|\mathbf{v}_\theta(\mathbf{x}_s, s) - \underbrace{(\boldsymbol{\epsilon} - \mathbf{x}_0)}_{\mathbf{v}_{\text{cond}}}\right\|_2^2\right]$$

This is the **simplest** generative model loss in the flow-matching framework:
just regress the network output against the conditional velocity.

Despite its simplicity, the marginal velocity $\mathbf{v}^*(\mathbf{x}_s, s)$ that minimises this loss
induces **curved** ODE trajectories (see Ch. 11 Fig. 2), which is why
multi-step ODE solvers are needed at sampling time.

### Why this is the foundation for everything else

The flow map models (CM, CTM, MF) all build on top of this velocity field:
- **CM**: jumps directly from $s$ to $0$ (bypasses the ODE)
- **CTM**: jumps from $s$ to any $t$ (generalised flow map)
- **MF**: learns the *average* velocity over $[t, s]$

The diffusion model *is* the ODE that these models try to shortcut.

In [ ]:
class DiffusionModel(nn.Module):
    """
    Diffusion model trained with Conditional Flow Matching (CFM).

    Learns a velocity network v_theta(x_s, s) that approximates the
    marginal velocity field v*(x_s, s) = E[eps - x_0 | x_s].

    Loss: ||v_theta(x_s, s) - (eps - x_0)||^2
    """
    def __init__(self, data_dim=2):
        super().__init__()
        self.net = MLP(data_dim=data_dim, n_time_conds=1)

    def v_theta(self, x_s, s):
        """Predict velocity v_theta(x_s, s)."""
        return self.net(x_s, s)

    def training_loss(self, x_0):
        B = x_0.shape[0]; dev = x_0.device
        eps = torch.randn_like(x_0)

        # LogNormal time sampling (concentrates on informative noise levels)
        log_s = -1.0 + 1.5 * torch.randn(B, device=dev)
        s = log_s.exp().clamp(1e-3, 1.0)

        x_s = forward_process(x_0, eps, s.unsqueeze(-1))
        v_cond = eps - x_0   # conditional velocity (ground truth)

        v_pred = self.v_theta(x_s, s)
        return (v_pred - v_cond).pow(2).mean()

## 4. Sampling: Euler Method (1st order)

Starting from $\mathbf{x}_1 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, solve the ODE $\frac{\mathrm{d}\mathbf{x}}{\mathrm{d}s} = \mathbf{v}_\theta(\mathbf{x}, s)$ backward from $s = 1$ to $s = 0$.

**Euler method** — one function evaluation per step:

$$\mathbf{x}_{s - \Delta s} = \mathbf{x}_s + (s_{\text{next}} - s)\,\mathbf{v}_\theta(\mathbf{x}_s, s) = \mathbf{x}_s - \Delta s\,\mathbf{v}_\theta(\mathbf{x}_s, s)$$

**Cost**: $N$ NFE for $N$ steps. Simple but accumulates discretisation error on curved trajectories.

## 5. Sampling: Heun's Method (2nd order)

**Heun's method** (improved Euler / explicit trapezoidal rule) — two function evaluations per step:

$$\tilde{\mathbf{x}} = \mathbf{x}_s + \Delta t\;\mathbf{v}_\theta(\mathbf{x}_s, s) \qquad \text{(predictor: Euler step)}$$

$$\mathbf{x}_{s + \Delta t} = \mathbf{x}_s + \frac{\Delta t}{2}\!\left(\mathbf{v}_\theta(\mathbf{x}_s, s) + \mathbf{v}_\theta(\tilde{\mathbf{x}}, s + \Delta t)\right) \qquad \text{(corrector: average slopes)}$$

where $\Delta t = s_{\text{next}} - s < 0$ (stepping backward).

**Cost**: $2N$ NFE for $N$ steps. But the 2nd-order accuracy means far fewer steps are needed to achieve the same quality as Euler — often $N_{\text{Heun}} \approx N_{\text{Euler}} / 4$.

**Last step special case**: when $s_{\text{next}} = 0$, we skip the corrector and use a single Euler step (following EDM convention). Evaluating $\mathbf{v}_\theta(\tilde{\mathbf{x}}, 0)$ at exactly $s=0$ can be numerically unstable, and the correction provides negligible benefit for the final step.

In [ ]:
@torch.no_grad()
def sample_euler(model, n_samples=5000, steps=100, data_dim=2):
    """
    Euler method (1st order): 1 NFE per step.

    x_{next} = x_s + dt * v_theta(x_s, s),    dt = s_next - s < 0
    """
    dev = next(model.parameters()).device
    model.eval()

    # Time schedule: 1.0 -> 0.0 in `steps` intervals
    ts = torch.linspace(1.0, 0.0, steps + 1, device=dev)
    x = torch.randn(n_samples, data_dim, device=dev)

    for i in range(steps):
        s_cur = ts[i]
        dt = ts[i + 1] - ts[i]   # negative (stepping backward)
        s_vec = torch.full((n_samples,), s_cur, device=dev)
        v = model.v_theta(x, s_vec)
        x = x + dt * v

    return x


@torch.no_grad()
def sample_heun(model, n_samples=5000, steps=50, data_dim=2):
    """
    Heun's method (2nd order, explicit trapezoidal): 2 NFE per step.

    Predictor (Euler):
        x_tilde = x_s + dt * v_theta(x_s, s)

    Corrector (average slopes):
        x_{next} = x_s + dt/2 * (v_theta(x_s, s) + v_theta(x_tilde, s_next))

    Last step (s_next = 0): use plain Euler (skip corrector at s=0).
    Total NFE = 2*steps - 1.
    """
    dev = next(model.parameters()).device
    model.eval()

    ts = torch.linspace(1.0, 0.0, steps + 1, device=dev)
    x = torch.randn(n_samples, data_dim, device=dev)

    for i in range(steps):
        s_cur  = ts[i]
        s_next = ts[i + 1]
        dt = s_next - s_cur   # negative

        s_vec = torch.full((n_samples,), s_cur, device=dev)
        v1 = model.v_theta(x, s_vec)

        # Last step or s_next very close to 0: plain Euler (skip corrector)
        if s_next <= 1e-6:
            x = x + dt * v1
        else:
            # Predictor: Euler step
            x_tilde = x + dt * v1
            # Corrector: evaluate at predicted point, average slopes
            s_next_vec = torch.full((n_samples,), s_next, device=dev)
            v2 = model.v_theta(x_tilde, s_next_vec)
            x = x + dt * 0.5 * (v1 + v2)

    return x

## 6. Training Loop

In [ ]:
def train_model(model, dataset, total_steps=10_000, batch_size=2048, lr=1e-3, device='cpu'):
    model = model.to(device); dataset = dataset.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []; model.train()
    for step in range(total_steps):
        idx = torch.randint(0, len(dataset), (batch_size,), device=device)
        x_0 = dataset[idx]
        loss = model.training_loss(x_0)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if (step + 1) % 2000 == 0 or step == 0:
            print(f'  step {step+1:>5d}/{total_steps}  loss = {loss.item():.4f}')
    model.eval(); return losses

## 7. Visualisation Helpers

In [ ]:
def plot_samples(data_np, samples_dict, title='', figsize_per=4.5):
    n = len(samples_dict)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, 4))
    if n == 1: axes = [axes]
    for ax, (label, pts) in zip(axes, samples_dict.items()):
        ax.scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.12, c='gray', label='data')
        ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.45, label=label)
        ax.set_title(label, fontsize=10)
        ax.set_xlim(-5, 7); ax.set_ylim(-4, 4); ax.set_aspect('equal')
        ax.legend(fontsize=7, loc='upper left')
    if title: fig.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
    fig.tight_layout(); plt.show()

def plot_losses(losses, title='Training Loss'):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(losses, alpha=0.8, linewidth=0.7)
    ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log')
    ax.set_title(title); fig.tight_layout(); plt.show()

## 8. Train the Diffusion Model

~30s–1min on Colab GPU.

In [ ]:
STEPS = 10_000; BS = 2048; LR = 1e-3; N = 5000
data_np = dataset.numpy()

print('=' * 50)
print('  Training Diffusion Model (CFM)')
print('=' * 50)
dm = DiffusionModel()
t0 = time.time()
dm_losses = train_model(dm, dataset, STEPS, BS, LR, device)
dm_time = time.time() - t0
print(f'  Training time: {dm_time:.1f}s')

plot_losses(dm_losses, title='Diffusion Model (CFM) Training Loss')

## 9. Euler vs Heun: Varying Number of Steps

We compare Euler ($N$ NFE) and Heun ($2N$ NFE) at the same number of **steps** $N$.

Heun uses 2x the function evaluations but achieves much better quality at low step counts.

In [ ]:
step_counts = [5, 10, 20, 50, 100, 500]

for n_steps in step_counts:
    euler_pts = sample_euler(dm, N, steps=n_steps).cpu().numpy()
    heun_pts  = sample_heun(dm, N, steps=n_steps).cpu().numpy()

    plot_samples(data_np, {
        f'Euler ({n_steps} steps, {n_steps} NFE)': euler_pts,
        f'Heun ({n_steps} steps, {2*n_steps-1} NFE)': heun_pts,
    }, title=f'{n_steps} Steps: Euler vs Heun')

## 10. Fair Comparison: Same NFE Budget

For a fair comparison, we match the **total NFE** (number of function evaluations):
- Euler with $N$ steps = $N$ NFE
- Heun with $N$ steps = $2N$ NFE (but the last step is 1 NFE, so actually $2N - 1$)

So Euler-$2N$ vs Heun-$N$ is roughly the same compute.

In [ ]:
# Fair comparison: match total NFE
nfe_budgets = [10, 20, 50, 100]

for nfe in nfe_budgets:
    euler_pts = sample_euler(dm, N, steps=nfe).cpu().numpy()           # nfe NFE
    heun_steps = (nfe + 1) // 2  # Heun uses ~2*steps NFE
    heun_pts  = sample_heun(dm, N, steps=heun_steps).cpu().numpy()    # ~nfe NFE

    plot_samples(data_np, {
        f'Euler ({nfe} steps = {nfe} NFE)': euler_pts,
        f'Heun ({heun_steps} steps = {2*heun_steps-1} NFE)': heun_pts,
    }, title=f'~{nfe} NFE Budget: Euler vs Heun')

## 11. Interactive Explorer

Drag sliders to compare Euler vs Heun at different step counts.

*(Requires ipywidgets — works in Colab/Jupyter)*

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def interactive_sample(solver, steps):
        if solver == 'Euler':
            pts = sample_euler(dm, 5000, steps=steps).cpu().numpy()
            nfe = steps
        else:
            pts = sample_heun(dm, 5000, steps=steps).cpu().numpy()
            nfe = 2 * steps - 1
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        ax.scatter(data_np[:5000, 0], data_np[:5000, 1], s=1, alpha=0.12, c='gray', label='data')
        lbl = f'{solver} | {steps} steps | {nfe} NFE'
        ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5, label=lbl)
        ax.set_xlim(-5, 7); ax.set_ylim(-4, 4); ax.set_aspect('equal')
        ax.legend(fontsize=9); ax.set_title(lbl)
        fig.tight_layout(); plt.show()

    w_solver = widgets.Dropdown(options=['Euler', 'Heun'], value='Euler', description='Solver:')
    w_steps  = widgets.IntSlider(min=1, max=500, value=50, description='Steps:')

    out = widgets.interactive_output(interactive_sample,
              {'solver': w_solver, 'steps': w_steps})
    display(widgets.VBox([widgets.HBox([w_solver, w_steps]), out]))

except ImportError:
    print('ipywidgets not available. Run in Colab/Jupyter for interactive sliders.')

## Summary

### Training

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{\mathbf{x}_0, \boldsymbol{\epsilon}, s}\!\left[\left\|\mathbf{v}_\theta(\mathbf{x}_s, s) - (\boldsymbol{\epsilon} - \mathbf{x}_0)\right\|_2^2\right]$$

This is the **simplest** loss in the flow-matching family. One network, one time condition, one regression target.

### Sampling

| Solver | Formula | NFE per step | Order | Best for |
|:-------|:--------|:-------------|:------|:---------|
| **Euler** | $\mathbf{x}_{s+\Delta t} = \mathbf{x}_s + \Delta t\,\mathbf{v}_\theta(\mathbf{x}_s, s)$ | 1 | 1st | Many steps ($\ge 100$) |
| **Heun** | Predictor + corrector (average slopes) | 2 | 2nd | Few steps ($10$–$50$) |

### Relationship to flow map models

| Model | Training | Sampling | Min steps for good quality |
|:------|:---------|:---------|:---------------------------|
| **Diffusion** (this notebook) | CFM loss | ODE solver (Euler/Heun) | ~50–1000 |
| **CM** | Consistency loss | Direct jump to $\mathbf{x}_0$ | **1** (but $\gamma\!=\!1$ only) |
| **CTM** | Semigroup loss + $\mathcal{L}_{\text{DM}}$ | Direct jump + $\gamma$-sampling | **1** |
| **MF** | MeanFlow Identity | Direct jump + $\gamma$-sampling | **1** |

The flow map models trade a more complex training procedure for dramatically fewer sampling steps.